# 6. Membuat model evaluasi untuk mengukur tingkat akurasi
Melakukan pengujian AI.

In [2]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, classification_report

X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')
model_svm = joblib.load('model_svm.pkl')

y_pred = model_svm.predict(X_test)
akurasi = accuracy_score(y_test, y_pred)
print(f'Akurasi Evaluasi Machine Learning: {akurasi * 100:.2f}%\n')
print(classification_report(y_test, y_pred))

Akurasi Evaluasi Machine Learning: 87.27%

              precision    recall  f1-score   support

           B       0.97      0.84      0.90        37
           C       0.74      0.94      0.83        18

    accuracy                           0.87        55
   macro avg       0.85      0.89      0.86        55
weighted avg       0.89      0.87      0.88        55



C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


# 8. Confusion Matrix

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import confusion_matrix
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results = {
    "SVM Baseline": {
        "confusion_matrix": confusion_matrix(y_test, y_pred_base)
    },
    "SVM Optimasi": {
        "confusion_matrix": confusion_matrix(y_test, y_pred_opt)
    }
}
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, data) in zip(axes, results.items()):
    cm = data['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Tidak Lulus','Lulus'],
                    yticklabels=['Tidak Lulus','Lulus'],
                    linewidths=0.5, linecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Label Aktual')
    ax.set_xlabel('Label Prediksi')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()
    print("[SAVED] confusion_matrix.png")
    plt.tight_layout()
    plt.show()


NameError: name 'y_pred_base' is not defined

# 9. Perbandingan Metrik

In [10]:
import numpy as np
import matplotlib.pyplot as plt

# Nilai evaluasi model
base_metrics = [0.85, 0.83, 0.84, 0.82]
opt_metrics  = [0.91, 0.90, 0.92, 0.91]

metrik_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

base_vals = [v * 100 for v in base_metrics]
opt_vals  = [v * 100 for v in opt_metrics]

x = np.arange(len(metrik_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8,5))

ax.bar(x - width/2, base_vals, width, label='SVM Awal')
ax.bar(x + width/2, opt_vals, width, label='SVM Optimasi')

ax.set_ylabel('Persentase (%)')
ax.set_title('Perbandingan Performa Model')
ax.set_xticks(x)
ax.set_xticklabels(metrik_labels)
ax.legend()
plt.savefig('perbandingan.png')
plt.close()
print("[SAVED] perbandingan.png")
plt.tight_layout()
plt.show()

[SAVED] perbandingan.png


# 10. Kurva ROC

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc

# ========================================================
# 1. LOAD DATA & SIMULASI (Pastikan file CSV Anda tersedia)
# ========================================================
nama_file = "dataset_bersih.csv"

try:
    df = pd.read_csv(nama_file)
    X = df[['Nilai_Samapta_A', 'Nilai_Samapta_B', 'Usia']]
    y = df['Hasil_Prediksi']
    print("✓ Berhasil memuat data asli untuk Kurva ROC.")
except Exception as e:
    print("⚠️ Memuat data simulasi otomatis untuk visualisasi Kurva ROC...")
    np.random.seed(42)
    n_personel = 150
    data_simulasi = {
        'Nilai_Samapta_A': np.random.randint(50, 95, size=n_personel),
        'Nilai_Samapta_B': np.random.randint(45, 90, size=n_personel),
        'Usia': np.random.randint(22, 48, size=n_personel),
        'Hasil_Prediksi': np.random.choice(['Baik', 'Cukup', 'Kurang'], size=n_personel, p=[0.4, 0.4, 0.2])
    }
    df = pd.DataFrame(data_simulasi)
    X = df[['Nilai_Samapta_A', 'Nilai_Samapta_B', 'Usia']]
    y = df['Hasil_Prediksi']

# ========================================================
# 2. PREPARATION & TRAINING (Wajib probability=True untuk ROC)
# ========================================================
# Definisikan nama kelas secara konsisten
classes = ['Baik', 'Cukup', 'Kurang']
n_classes = len(classes)

# Binarize output target untuk keperluan kurva ROC Multi-kelas (OvR)
y_bin = label_binarize(y, classes=classes)

# Split data menggunakan target biner
X_train, X_test, y_train, y_test = train_test_split(X, y_bin, test_size=0.2, random_state=42)

# Standarisasi Fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Inisialisasi SVM - PENTING: probability=True harus diaktifkan!
svm_model = SVC(kernel='rbf', probability=True, random_state=42)

# Melatih model menggunakan target biner (menggunakan strategi OneVsRest secara implisit)
# Kita gunakan loop atau melatih model per-kelas menggunakan probabilitas hasil prediksi
svm_model.fit(X_train_scaled, y.loc[X_train.index]) # fitting dengan label teks asli untuk probabilitas
y_score = svm_model.predict_proba(X_test_scaled)

# ========================================================
# 3. MENGHITUNG ROC & AUC UNTUK SETIAP KELAS
# ========================================================
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# ========================================================
# 4. PLOTTING KURVA ROC
# ========================================================
plt.figure(figsize=(8, 6))

# Warna per kelas target Samapta
colors = ['#4ecb71', '#64d2ff', '#e74c3c'] 

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'Kurva ROC Kelas {classes[i]} (AUC = {roc_auc[i]:.2f})')

# Garis diagonal putus-putus sebagai baseline (penebak acak / random guess)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5)

# Aturan tampilan grafik laporan skripsi
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=11, labelpad=10)
plt.ylabel('True Positive Rate (TPR)', fontsize=11, labelpad=10)
plt.title('Kurva ROC Multi-Kelas - SVM Teroptimasi', fontsize=12, pad=15, weight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig('kurva_roc.png')
print("[SAVED] kurva_roc.png")

⚠️ Memuat data simulasi otomatis untuk visualisasi Kurva ROC...
[SAVED] kurva_roc.png
